# Step 5: Critical Thinking, Ethical AI & Bias Auditing
**Capstone Project — Cybersecurity: Detect Anomalies in Network Traffic**

Dataset: CICIDS2017  
This notebook covers:
1. SHAP explainability — global (beeswarm, bar) and local (waterfall, force plots)
2. LIME — local surrogate explanations for individual predictions
3. PDP + ICE — partial dependence and individual conditional expectation plots
4. Limitations — class imbalance impact, overfitting analysis, leakage risk, temporal drift
5. Bias audit — proxy group definitions, fairness metrics (FPR/FNR parity, disparate impact, equalized odds)
6. Mitigation strategies — threshold tuning, cost-sensitive retraining, monitoring

## 1. Setup & Imports

In [ ]:
!pip install shap lime scikit-learn xgboost matplotlib seaborn pandas numpy joblib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
import json

warnings.filterwarnings('ignore')

import joblib
import shap
import lime
import lime.lime_tabular

from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

os.makedirs('./models', exist_ok=True)
os.makedirs('./outputs/step5', exist_ok=True)

print('Imports complete.')

## 2. Load Models & Data

All artefacts are loaded from `./models/` — saved at the end of Step 4.
We re-apply the same 80/20 stratified split used in Step 4 so SHAP and fairness metrics
are computed on held-out test data only.

In [ ]:
# Load feature matrix and labels
X = np.load('./models/X_selected.npy')
y_bin = np.load('./models/y_binary.npy')
y_multi = np.load('./models/y_multiclass.npy')

feature_names = pd.read_csv('./models/selected_features.csv')['feature'].tolist()
category_labels = joblib.load('./models/y_category_labels.pkl')
class_names = sorted(set(category_labels))

print(f'X shape          : {X.shape}')
print(f'Features         : {len(feature_names)}')
print(f'Binary classes   : {dict(zip(*np.unique(y_bin, return_counts=True)))}')
print(f'Attack categories: {class_names}')

In [ ]:
# Reproduce the same train/test split from Step 4
X_train, X_test, y_train_bin, y_test_bin, y_train_multi, y_test_multi = train_test_split(
    X, y_bin, y_multi,
    test_size=0.20,
    stratify=y_bin,
    random_state=RANDOM_SEED
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

# Load trained models
xgb_bin   = joblib.load('./models/xgb_binary_tuned.pkl')
rf_bin    = joblib.load('./models/rf_binary_tuned.pkl')
lr_bin    = joblib.load('./models/lr_binary.pkl')
xgb_multi = joblib.load('./models/xgb_multiclass_tuned.pkl')
rf_multi  = joblib.load('./models/rf_multiclass_tuned.pkl')

# Load comparison table
df_comparison = pd.read_csv('./models/model_comparison.csv')

print('Models loaded.')

## 3. SHAP Explainability

SHAP (SHapley Additive exPlanations) provides a unified framework for attributing model predictions
to individual features. TreeSHAP computes exact Shapley values for tree-based models in polynomial
time, making it tractable on the full test set.

We use the tuned XGBoost (binary) as the primary explainability target because it is the
recommended production model from Step 4.

In [ ]:
# TreeSHAP explainer — exact, no approximation needed for XGBoost
explainer = shap.TreeExplainer(xgb_bin)

# Compute SHAP values on a stratified 2000-row test subsample for speed
rng = np.random.default_rng(RANDOM_SEED)
idx_benign  = np.where(y_test_bin == 0)[0]
idx_attack  = np.where(y_test_bin == 1)[0]

n_each = min(1000, len(idx_benign), len(idx_attack))
sample_idx = np.concatenate([
    rng.choice(idx_benign, n_each, replace=False),
    rng.choice(idx_attack, n_each, replace=False)
])
X_shap = X_test[sample_idx]
y_shap = y_test_bin[sample_idx]

shap_values = explainer(X_shap)

print(f'SHAP values shape: {shap_values.values.shape}')
print(f'Sample: {n_each} benign + {n_each} attack = {len(sample_idx)} total')

In [ ]:
# --- Global: Beeswarm plot ---
# Each dot = one sample; x = SHAP value; colour = feature value (red=high, blue=low)
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('SHAP Beeswarm — XGBoost Tuned (Binary)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('./outputs/step5/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/shap_beeswarm.png')

In [ ]:
# --- Global: Bar plot (mean |SHAP|) ---
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title('Mean |SHAP| Feature Importance — XGBoost Tuned (Binary)', fontsize=12)
plt.tight_layout()
plt.savefig('./outputs/step5/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/shap_bar.png')

# Extract top 10 feature names for use in PDP/ICE and fairness sections
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top_shap_idx  = np.argsort(mean_abs_shap)[::-1][:10]
top_features  = [feature_names[i] for i in top_shap_idx]
print(f'\nTop 10 features by mean |SHAP|:')
for rank, (idx, fname) in enumerate(zip(top_shap_idx, top_features), 1):
    print(f'  {rank:2d}. {fname:<40} mean|SHAP| = {mean_abs_shap[idx]:.4f}')

In [ ]:
# --- Local: Waterfall plots ---
# Pick one correctly classified benign and one correctly classified attack instance

y_pred_shap = xgb_bin.predict(X_shap)

correct_benign = np.where((y_shap == 0) & (y_pred_shap == 0))[0]
correct_attack = np.where((y_shap == 1) & (y_pred_shap == 1))[0]
misclassified  = np.where(y_shap != y_pred_shap)[0]

for label, candidate_idx in [
    ('Correctly Classified — Benign', correct_benign[0]),
    ('Correctly Classified — Attack', correct_attack[0]),
]:
    fig = plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_values[candidate_idx], max_display=15, show=False)
    plt.title(f'SHAP Waterfall: {label}', fontsize=11)
    plt.tight_layout()
    fname = label.lower().replace(' ', '_').replace('—', '').replace('  ', '_') + '.png'
    plt.savefig(f'./outputs/step5/shap_waterfall_{fname}', dpi=150, bbox_inches='tight')
    plt.show()

# Misclassified instance (if any)
if len(misclassified) > 0:
    fig = plt.figure(figsize=(10, 6))
    true_lbl  = 'Benign' if y_shap[misclassified[0]] == 0 else 'Attack'
    pred_lbl  = 'Benign' if y_pred_shap[misclassified[0]] == 0 else 'Attack'
    shap.plots.waterfall(shap_values[misclassified[0]], max_display=15, show=False)
    plt.title(f'SHAP Waterfall: Misclassified — True={true_lbl}, Pred={pred_lbl}', fontsize=11)
    plt.tight_layout()
    plt.savefig('./outputs/step5/shap_waterfall_misclassified.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Misclassified instance: true={true_lbl}, pred={pred_lbl}')
else:
    print('No misclassified instances in SHAP sample.')

In [ ]:
# --- Dependence plots for top 3 features ---
# Shows how SHAP value for a feature varies with its raw value;
# colour encodes the strongest interacting feature automatically.

top3 = top_shap_idx[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat_idx in zip(axes, top3):
    shap.plots.scatter(
        shap_values[:, feat_idx],
        color=shap_values,
        ax=ax,
        show=False
    )
    ax.set_title(f'{feature_names[feat_idx]}', fontsize=10)

plt.suptitle('SHAP Dependence Plots — Top 3 Features', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('./outputs/step5/shap_dependence_top3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/shap_dependence_top3.png')

## 4. LIME — Local Surrogate Explanations

LIME (Local Interpretable Model-Agnostic Explanations) fits a simple linear model in the
neighbourhood of each prediction to approximate how the black-box model behaves locally.
Unlike SHAP, LIME is model-agnostic and useful for cross-checking SHAP attributions on
specific instances of interest.

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=['Benign', 'Attack'],
    mode='classification',
    discretize_continuous=True,
    random_state=RANDOM_SEED
)

def explain_with_lime(instance_idx_in_shap_sample, label='instance'):
    instance = X_shap[instance_idx_in_shap_sample]
    exp = lime_explainer.explain_instance(
        instance,
        xgb_bin.predict_proba,
        num_features=15,
        top_labels=1
    )
    fig = exp.as_pyplot_figure(label=1)  # class 1 = Attack
    fig.set_size_inches(10, 6)
    plt.title(f'LIME Explanation — {label} (Attack class probability)', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'./outputs/step5/lime_{label.lower().replace(" ", "_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    pred_prob = xgb_bin.predict_proba(instance.reshape(1, -1))[0, 1]
    true_lbl  = 'Attack' if y_shap[instance_idx_in_shap_sample] == 1 else 'Benign'
    print(f'  True label: {true_lbl}  |  P(Attack): {pred_prob:.4f}')
    return exp

print('LIME explainer ready.')

In [ ]:
# Explain a correctly classified benign flow
exp_benign = explain_with_lime(correct_benign[0], label='Correctly Classified Benign')

In [ ]:
# Explain a correctly classified attack flow
exp_attack = explain_with_lime(correct_attack[0], label='Correctly Classified Attack')

In [ ]:
# Explain a misclassified instance to understand model failure modes
if len(misclassified) > 0:
    exp_mis = explain_with_lime(misclassified[0], label='Misclassified Instance')
else:
    print('No misclassifications in SHAP sample — model performed perfectly on this subset.')

In [ ]:
# Agreement check: top-5 features from SHAP vs LIME on the attack instance
lime_weights = dict(exp_attack.as_list(label=1))
lime_top5 = sorted(lime_weights, key=lambda k: abs(lime_weights[k]), reverse=True)[:5]

shap_top5 = top_features[:5]

print('=== SHAP vs LIME agreement on attack instance ===')
print(f'SHAP top 5 (global): {shap_top5}')
print(f'LIME top 5 (local) : {lime_top5}')

overlap = set(shap_top5) & set([f.split(' ')[0] for f in lime_top5])
print(f'Overlap in top-5   : {len(overlap)} features')

## 5. Partial Dependence Plots (PDP) + Individual Conditional Expectation (ICE)

PDP shows the marginal effect of a feature on the predicted probability, averaged across
all other features. ICE plots show the same relationship for individual samples, revealing
whether the average trend masks heterogeneous effects across subpopulations.

We plot PDP + ICE for the top 4 features from SHAP analysis.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
from sklearn.ensemble import RandomForestClassifier

# Use Random Forest for PDP/ICE — sklearn's PartialDependenceDisplay has native support
# XGBoost works too but RF gives cleaner probability calibration for these plots

top4_idx = top_shap_idx[:4].tolist()
top4_names = [feature_names[i] for i in top4_idx]

print(f'PDP/ICE for features: {top4_names}')

# Use a 5000-row subsample for ICE (to avoid overplotting)
rng2 = np.random.default_rng(RANDOM_SEED + 1)
pdp_idx = rng2.choice(len(X_test), min(5000, len(X_test)), replace=False)
X_pdp = X_test[pdp_idx]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat_idx, feat_name in zip(axes.flatten(), top4_idx, top4_names):
    disp = PartialDependenceDisplay.from_estimator(
        rf_bin,
        X_pdp,
        features=[feat_idx],
        feature_names=feature_names,
        kind='both',          # PDP + ICE
        subsample=200,        # number of ICE lines
        n_jobs=-1,
        random_state=RANDOM_SEED,
        ice_lines_kw={'color': 'lightblue', 'alpha': 0.3, 'linewidth': 0.5},
        pd_line_kw={'color': 'firebrick', 'linewidth': 2.5},
        ax=ax
    )
    ax.set_title(f'PDP + ICE: {feat_name}', fontsize=10)
    ax.set_ylabel('P(Attack)')

plt.suptitle('Partial Dependence + ICE Plots — RF Tuned (Binary)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('./outputs/step5/pdp_ice_top4.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/pdp_ice_top4.png')

## 6. Limitations Analysis

This section documents the key limitations of the trained models. Honest identification
of limitations is a prerequisite for responsible deployment.

### 6.1 Overfitting — Train vs Test Performance Gap

In [ ]:
models_to_check = {
    'Logistic Regression': lr_bin,
    'RF Tuned':            rf_bin,
    'XGBoost Tuned':       xgb_bin
}

overfit_rows = []
for name, model in models_to_check.items():
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    train_f1 = f1_score(y_train_bin, y_pred_train, average='weighted')
    test_f1  = f1_score(y_test_bin,  y_pred_test,  average='weighted')
    train_rec = recall_score(y_train_bin, y_pred_train)
    test_rec  = recall_score(y_test_bin,  y_pred_test)

    overfit_rows.append({
        'Model': name,
        'Train F1 (wtd)': round(train_f1, 4),
        'Test F1 (wtd)':  round(test_f1,  4),
        'F1 Gap':         round(train_f1 - test_f1, 4),
        'Train Recall':   round(train_rec, 4),
        'Test Recall':    round(test_rec,  4),
        'Recall Gap':     round(train_rec - test_rec, 4)
    })

df_overfit = pd.DataFrame(overfit_rows).set_index('Model')
print('=== Overfitting Analysis: Train vs Test ===')
print(df_overfit.to_string())
df_overfit.to_csv('./outputs/step5/overfit_analysis.csv')

print('\nInterpretation:')
for row in overfit_rows:
    gap = row['F1 Gap']
    level = 'Minimal' if gap < 0.01 else ('Moderate' if gap < 0.05 else 'High')
    print(f"  {row['Model']}: F1 gap = {gap:.4f} ({level} overfitting)")

### 6.2 Class Imbalance Impact

In [ ]:
# Per-class recall and precision for the multiclass XGBoost model
y_pred_multi_xgb = xgb_multi.predict(X_test)

from sklearn.metrics import classification_report
report = classification_report(
    y_test_multi, y_pred_multi_xgb,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report).T.iloc[:-3]  # drop avg rows
df_report = df_report[['precision', 'recall', 'f1-score', 'support']].copy()
df_report['support'] = df_report['support'].astype(int)

print('=== Per-Class Metrics — XGBoost Multiclass ===')
print(df_report.to_string(float_format='{:.4f}'.format))

# Visualise recall by class size
fig, ax = plt.subplots(figsize=(10, 5))
ax2 = ax.twinx()

x = np.arange(len(df_report))
ax.bar(x - 0.2, df_report['precision'], 0.35, label='Precision', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, df_report['recall'],    0.35, label='Recall',    color='firebrick',  alpha=0.8)
ax2.plot(x, df_report['support'], 'ko--', linewidth=1.5, markersize=5, label='Support (right axis)')

ax.set_xticks(x)
ax.set_xticklabels(df_report.index, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Precision / Recall')
ax2.set_ylabel('Support (# test samples)')
ax.set_title('Per-Class Performance vs Class Size — XGBoost Multiclass', fontsize=11)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('./outputs/step5/class_imbalance_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/class_imbalance_impact.png')

print('\nKey observation:')
low_recall = df_report[df_report['recall'] < 0.5]
if not low_recall.empty:
    for cls in low_recall.index:
        print(f'  {cls}: recall={df_report.loc[cls,"recall"]:.3f}, support={df_report.loc[cls,"support"]}')
else:
    print('  All classes achieve recall >= 0.50.')

### 6.3 Feature Leakage Risk Assessment

In [ ]:
# Identify features from SHAP that are high-risk for leakage in live deployment
# Leakage risk categories for CICFlowMeter features:
#   HIGH: features computed using the full flow (require observing the complete connection)
#   MED : features stable mid-flow but potentially delayed
#   LOW : observable from early packets (safe for real-time use)

leakage_risk = {
    # Rate features require full flow duration
    'Flow Bytes/s':           'HIGH - needs full flow duration',
    'Flow Packets/s':         'HIGH - needs full flow duration',
    # Duration itself is end-of-flow
    'Flow Duration':          'HIGH - only known at flow end',
    # Backward stats only available after receiving replies
    'Bwd Packet Length Max':  'MED  - needs backward packets',
    'Bwd Packet Length Mean': 'MED  - needs backward packets',
    'Bwd IAT Total':          'MED  - needs backward timing',
    # Forward stats available early
    'Total Fwd Packets':      'LOW  - accumulates during flow',
    'SYN Flag Count':         'LOW  - visible from first SYN',
    'Destination Port':       'LOW  - in first packet',
    'Protocol':               'LOW  - in first packet',
}

print('=== Feature Leakage Risk Assessment ===')
print(f'{"Feature":<35} {"Risk Level"}')
print('-' * 70)

risk_in_top10 = {}
for feat in top_features:
    # fuzzy match
    matched = next((k for k in leakage_risk if k.lower() in feat.lower() or feat.lower() in k.lower()), None)
    if matched:
        risk_in_top10[feat] = leakage_risk[matched]
        print(f'{feat:<35} {leakage_risk[matched]}')

high_risk = [f for f, r in risk_in_top10.items() if r.startswith('HIGH')]
print(f'\nHigh-leakage features in top 10: {len(high_risk)}')
print(f'Note: Flow-level features like Flow Bytes/s and Flow Duration are standard in')
print(f'network IDS research. In production, flows would be processed at connection close.')
print(f'For real-time detection, an online variant with sub-flow windowing is recommended.')

### 6.4 Temporal Drift & Adversarial Evasion Risk

In [ ]:
print('=== Temporal Drift & Adversarial Risk Assessment ===')
print()
print('1. TEMPORAL DRIFT')
print('   CICIDS2017 captures traffic from July 2017. Network protocols, attack tooling,')
print('   and traffic volumes have evolved substantially. Key degradation risks:')
print('   - New attack variants (e.g., HTTP/2 DoS, QUIC-based DDoS) not in training data')
print('   - Shifts in baseline traffic (TLS 1.3 adoption changes packet size distributions)')
print('   - Botnet C2 channels have moved from IRC to encrypted HTTPS')
print()
print('   Mitigation: monthly retraining on rolling windows of labeled production traffic;')
print('   Population Stability Index (PSI) monitoring on input feature distributions.')
print()
print('2. ADVERSARIAL EVASION')
print('   Attackers aware of the model can manipulate flow statistics to evade detection:')
print('   - Rate-limiting attack traffic to fall within benign byte-rate ranges')
print('   - Padding packets to shift size distributions toward benign baselines')
print('   - Fragmenting attacks into small sub-flows below detection thresholds')
print()
print('   Mitigation: ensemble of diverse model types (tree + neural); anomaly detection')
print('   layer for out-of-distribution inputs; adversarial training with perturbed examples.')
print()
print('3. LABEL QUALITY')
print('   CICIDS2017 labels were generated from attack scripts against known targets.')
print('   Some benign flows may be mislabeled (researchers have documented ~5% label noise).')
print('   Impact: inflates false positive rate estimates; model may have learned noise patterns.')

## 7. Bias Audit — Proxy Group Analysis

CICIDS2017 does not contain traditional sensitive attributes (age, gender, race).
In a network traffic context, the analogous concern is **operational fairness**: does the
model produce systematically different error rates for distinct traffic segments?

Unequal False Positive Rates (FPR) across traffic segments have concrete operational
consequences. A segment with elevated FPR generates more analyst workload and may lead
to blocking legitimate business traffic. Unequal FNR means certain traffic types are
systematically under-protected.

**Proxy groups defined** (from the selected feature set):
- **Protocol**: TCP vs UDP vs Other (derived from raw protocol value if available)
- **Traffic Volume Quartile**: Q1/Q2/Q3/Q4 by total forward byte volume
- **Port Category**: Well-known (0-1023) vs Registered (1024-49151) vs Ephemeral (49152+)
- **Flow Duration Bin**: Zero-duration vs Short vs Medium vs Long

In [ ]:
# Build a DataFrame of the test set with feature values and predictions
df_test = pd.DataFrame(X_test, columns=feature_names)
df_test['y_true'] = y_test_bin
df_test['y_pred'] = xgb_bin.predict(X_test)
df_test['y_prob'] = xgb_bin.predict_proba(X_test)[:, 1]

print(f'Test set shape: {df_test.shape}')
print(f'Positive rate in test: {y_test_bin.mean():.4f}')

# --- Group 1: Traffic Volume Quartile ---
# Use first feature related to bytes/packets volume (log-transformed in Step 3)
vol_feat = next((f for f in feature_names if 'bytes' in f.lower() or 'fwd' in f.lower()), feature_names[0])
print(f'\nVolume proxy feature: {vol_feat}')
df_test['volume_quartile'] = pd.qcut(
    df_test[vol_feat], q=4,
    labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'],
    duplicates='drop'
)

# --- Group 2: Flow Duration Bin ---
dur_feat = next((f for f in feature_names if 'duration' in f.lower()), None)
if dur_feat:
    print(f'Duration proxy feature: {dur_feat}')
    df_test['duration_bin'] = pd.cut(
        df_test[dur_feat],
        bins=[-np.inf, -1.5, -0.5, 0.5, np.inf],
        labels=['Very Short', 'Short', 'Medium', 'Long']
    )
else:
    print('Duration feature not found in selected set — skipping duration group.')
    df_test['duration_bin'] = 'N/A'

# --- Group 3: Protocol (if TCP/UDP flags present) ---
has_tcp = any('tcp' in f.lower() for f in feature_names)
has_udp = any('udp' in f.lower() for f in feature_names)
protocol_feat = next((f for f in feature_names if 'protocol' in f.lower()), None)

if protocol_feat:
    print(f'Protocol proxy feature: {protocol_feat}')
    df_test['protocol_group'] = df_test[protocol_feat].apply(
        lambda v: 'TCP' if v > 0.5 else ('UDP' if v < -0.5 else 'Other')
    )
elif has_tcp or has_udp:
    tcp_feat = next((f for f in feature_names if 'tcp' in f.lower()), None)
    udp_feat = next((f for f in feature_names if 'udp' in f.lower()), None)
    if tcp_feat and udp_feat:
        df_test['protocol_group'] = np.where(
            df_test[tcp_feat] > 0, 'TCP',
            np.where(df_test[udp_feat] > 0, 'UDP', 'Other')
        )
    else:
        df_test['protocol_group'] = 'Unknown'
else:
    # Derive protocol group from SYN flag as TCP proxy
    syn_feat = next((f for f in feature_names if 'syn' in f.lower()), None)
    if syn_feat:
        df_test['protocol_group'] = df_test[syn_feat].apply(
            lambda v: 'TCP-like' if v > 0 else 'Non-TCP'
        )
        print(f'Protocol proxy via SYN flag: {syn_feat}')
    else:
        df_test['protocol_group'] = 'Unavailable'
        print('Protocol proxy not available in selected feature set.')

print('\nGroup distributions:')
for col in ['volume_quartile', 'duration_bin', 'protocol_group']:
    if col in df_test.columns:
        print(f'  {col}: {dict(df_test[col].value_counts())}')

In [ ]:
def compute_group_fairness(df, group_col, label='y_true', pred='y_pred'):
    """
    Compute per-group fairness metrics:
    - Positive Prediction Rate (PPR) — demographic parity
    - False Positive Rate (FPR) = FP / (FP + TN)
    - False Negative Rate (FNR) = FN / (FN + TP)
    - Disparate Impact Ratio (DIR) = min(PPR_g) / max(PPR_g) — threshold: >= 0.80
    """
    rows = []
    for group_val, gdf in df.groupby(group_col):
        yt = gdf[label].values
        yp = gdf[pred].values

        if len(yt) < 20:  # skip tiny groups
            continue

        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel() if len(np.unique(yt)) > 1 else (0, 0, 0, 0)

        ppr = yp.mean()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        prec = precision_score(yt, yp, zero_division=0)
        rec  = recall_score(yt, yp, zero_division=0)
        f1   = f1_score(yt, yp, zero_division=0)

        rows.append({
            'Group': str(group_val),
            'N': len(yt),
            'Attack%': round(yt.mean() * 100, 1),
            'PPR': round(ppr, 4),
            'FPR': round(fpr, 4),
            'FNR': round(fnr, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'F1': round(f1, 4)
        })

    df_metrics = pd.DataFrame(rows)
    if len(df_metrics) > 0:
        max_ppr = df_metrics['PPR'].max()
        min_ppr = df_metrics['PPR'].min()
        dir_ratio = min_ppr / max_ppr if max_ppr > 0 else np.nan

        fpr_diff = df_metrics['FPR'].max() - df_metrics['FPR'].min()
        fnr_diff = df_metrics['FNR'].max() - df_metrics['FNR'].min()

        print(f'  Disparate Impact Ratio  : {dir_ratio:.4f} (threshold >= 0.80 for fairness)')
        print(f'  FPR range (max - min)   : {fpr_diff:.4f} (equalized odds criterion)')
        print(f'  FNR range (max - min)   : {fnr_diff:.4f} (equalized odds criterion)')

    return df_metrics


print('=== Fairness Metrics by Traffic Volume Quartile ===')
df_fair_volume = compute_group_fairness(df_test, 'volume_quartile')
print(df_fair_volume.to_string(index=False))

print('\n=== Fairness Metrics by Flow Duration Bin ===')
df_fair_duration = compute_group_fairness(df_test, 'duration_bin')
print(df_fair_duration.to_string(index=False))

print('\n=== Fairness Metrics by Protocol Group ===')
df_fair_protocol = compute_group_fairness(df_test, 'protocol_group')
print(df_fair_protocol.to_string(index=False))

In [ ]:
# Visualise fairness metrics as grouped heatmaps

def plot_fairness_heatmap(df_fair, title, filename):
    if df_fair.empty:
        print(f'No data for: {title}')
        return

    metric_cols = ['PPR', 'FPR', 'FNR', 'Precision', 'Recall', 'F1']
    heat_data = df_fair.set_index('Group')[metric_cols]

    fig, ax = plt.subplots(figsize=(10, max(3, len(heat_data) * 0.8 + 1)))
    sns.heatmap(
        heat_data.astype(float),
        annot=True, fmt='.3f',
        cmap='RdYlGn',
        vmin=0, vmax=1,
        linewidths=0.5,
        ax=ax
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Fairness Metric')
    ax.set_ylabel('Group')
    plt.tight_layout()
    plt.savefig(f'./outputs/step5/{filename}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: ./outputs/step5/{filename}')


plot_fairness_heatmap(df_fair_volume,   'Fairness by Traffic Volume Quartile', 'fairness_volume.png')
plot_fairness_heatmap(df_fair_duration, 'Fairness by Flow Duration Bin',       'fairness_duration.png')
plot_fairness_heatmap(df_fair_protocol, 'Fairness by Protocol Group',          'fairness_protocol.png')

In [ ]:
# FPR vs FNR scatter — Equalized Odds visualisation across all groups

all_groups = pd.concat([
    df_fair_volume.assign(GroupType='Volume'),
    df_fair_duration.assign(GroupType='Duration'),
    df_fair_protocol.assign(GroupType='Protocol')
], ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 6))

palette = {'Volume': 'steelblue', 'Duration': 'firebrick', 'Protocol': 'seagreen'}
for gtype, gdf in all_groups.groupby('GroupType'):
    ax.scatter(
        gdf['FPR'], gdf['FNR'],
        label=gtype, color=palette[gtype], s=80, zorder=3
    )
    for _, row in gdf.iterrows():
        ax.annotate(
            row['Group'], (row['FPR'], row['FNR']),
            fontsize=7, textcoords='offset points', xytext=(5, 3)
        )

# Ideal point
ax.scatter([0], [0], marker='*', color='gold', s=200, zorder=5, label='Ideal (FPR=0, FNR=0)')
ax.set_xlabel('False Positive Rate (FPR) — analyst alert load')
ax.set_ylabel('False Negative Rate (FNR) — missed attack rate')
ax.set_title('Equalized Odds: FPR vs FNR Across Traffic Groups', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('./outputs/step5/equalized_odds_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/equalized_odds_scatter.png')

In [ ]:
# Compute and print Disparate Impact Ratios for all groupings

print('=== Disparate Impact Summary ===')
print(f'(DIR = min group PPR / max group PPR — threshold >= 0.80 considered fair)')
print()

for label, df_fair in [
    ('Volume Quartile', df_fair_volume),
    ('Duration Bin',    df_fair_duration),
    ('Protocol Group',  df_fair_protocol)
]:
    if df_fair.empty:
        print(f'{label}: insufficient data')
        continue
    ppr_vals = df_fair['PPR'].dropna()
    if len(ppr_vals) < 2 or ppr_vals.max() == 0:
        print(f'{label}: cannot compute DIR (zero variance in PPR)')
        continue
    dir_val  = ppr_vals.min() / ppr_vals.max()
    fpr_diff = df_fair['FPR'].max() - df_fair['FPR'].min()
    fnr_diff = df_fair['FNR'].max() - df_fair['FNR'].min()
    status   = 'PASS' if dir_val >= 0.80 else 'FAIL'
    print(f'{label:<25} DIR={dir_val:.4f} [{status}]  FPR range={fpr_diff:.4f}  FNR range={fnr_diff:.4f}')

print()
print('Note: FAIL does not necessarily indicate discriminatory model behaviour.')
print('In a network IDS, attack traffic is not uniformly distributed across protocol/volume')
print('groups by design — attackers target specific ports and generate atypical flow volumes.')
print('The DIR metric must be interpreted in light of the base attack rate per group.')

## 8. Mitigation Strategies

For each identified limitation or bias finding, we propose a concrete, feasible mitigation.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

# --- 8.1 Threshold Optimisation ---
# Default threshold = 0.5 is not optimal under asymmetric costs
# We sweep thresholds and identify the operating point that achieves
# Recall >= 0.99 (target from Step 1) at maximum precision

y_prob_test = xgb_bin.predict_proba(X_test)[:, 1]

precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_test_bin, y_prob_test)
fpr_curve, tpr_curve, thresholds_roc = roc_curve(y_test_bin, y_prob_test)

# Find threshold achieving Recall >= 0.99
target_recall = 0.99
valid_idx = np.where(recall_curve[:-1] >= target_recall)[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precision_curve[valid_idx])]
    opt_threshold = thresholds_pr[best_idx]
    opt_precision = precision_curve[best_idx]
    opt_recall    = recall_curve[best_idx]
    print(f'Optimal threshold for Recall >= {target_recall}: {opt_threshold:.4f}')
    print(f'  Resulting Precision: {opt_precision:.4f}')
    print(f'  Resulting Recall   : {opt_recall:.4f}')
else:
    opt_threshold = 0.5
    print(f'Target recall {target_recall} not achievable — using default 0.5')

# Apply optimised threshold
y_pred_opt = (y_prob_test >= opt_threshold).astype(int)
f1_opt     = f1_score(y_test_bin, y_pred_opt, average='weighted')
rec_opt    = recall_score(y_test_bin, y_pred_opt)
prec_opt   = precision_score(y_test_bin, y_pred_opt, zero_division=0)
print(f'\nAt threshold={opt_threshold:.4f}: F1={f1_opt:.4f}, Recall={rec_opt:.4f}, Precision={prec_opt:.4f}')

# Plot Precision-Recall curve with operating points
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(recall_curve, precision_curve, color='steelblue', lw=2, label='PR Curve')
ax.scatter([opt_recall], [opt_precision], color='firebrick', s=100, zorder=5,
           label=f'Optimal t={opt_threshold:.3f} (Recall≥{target_recall})')
ax.scatter([rec_opt], [prec_opt], color='orange', s=80, zorder=4, marker='s',
           label=f'Default t=0.5')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — XGBoost Binary', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)

ax = axes[1]
ax.plot(fpr_curve, tpr_curve, color='steelblue', lw=2, label='ROC Curve')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — XGBoost Binary', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Threshold Optimisation — Operating Point Selection', fontsize=12)
plt.tight_layout()
plt.savefig('./outputs/step5/threshold_optimisation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./outputs/step5/threshold_optimisation.png')

In [ ]:
# --- 8.2 Per-Group Threshold Calibration ---
# To equalise FNR across volume quartile groups, we can set different thresholds per group

print('=== Per-Group Threshold Calibration (Volume Quartile) ===')
print('Goal: bring FNR within 0.02 of overall FNR across all volume groups\n')

overall_fnr = 1 - recall_score(y_test_bin, xgb_bin.predict(X_test))
print(f'Overall FNR at threshold=0.5: {overall_fnr:.4f}')

df_test['y_prob'] = y_prob_test

group_thresholds = {}
for group_val, gdf in df_test.groupby('volume_quartile'):
    yt = gdf['y_true'].values
    yp = gdf['y_prob'].values

    if len(np.unique(yt)) < 2 or yt.sum() < 5:
        group_thresholds[str(group_val)] = 0.5
        continue

    # Find threshold that minimises |group FNR - overall FNR|
    best_t, best_diff = 0.5, 1.0
    for t in np.arange(0.1, 0.9, 0.02):
        yp_t = (yp >= t).astype(int)
        if yp_t.sum() == 0:
            continue
        fnr_g = 1 - recall_score(yt, yp_t, zero_division=0)
        diff  = abs(fnr_g - overall_fnr)
        if diff < best_diff:
            best_diff, best_t = diff, t

    group_thresholds[str(group_val)] = best_t
    yp_best = (yp >= best_t).astype(int)
    fnr_best = 1 - recall_score(yt, yp_best, zero_division=0)
    fpr_best = (yp_best[yt == 0]).mean() if (yt == 0).sum() > 0 else np.nan
    print(f'  {str(group_val):<12}: optimal threshold={best_t:.2f}  '
          f'FNR={fnr_best:.4f}  FPR={fpr_best:.4f}')

print(f'\nGroup thresholds: {group_thresholds}')
print('\nNote: per-group thresholds require the group attribute to be known at inference time.')
print('This is feasible because traffic volume is observable from the flow itself.')

In [ ]:
print('=== Mitigation Strategy Summary ===')
print()
mitigations = [
    ('Class Imbalance',
     'Apply SMOTE or ADASYN in the training pipeline for Botnet and Infiltration classes.\n'
     '     Explore cost-sensitive XGBoost with per-class sample weights tuned via nested CV.\n'
     '     Consider a two-stage detector: first binary (Benign vs Attack), then multiclass.'),

    ('Overfitting',
     'Increase L1/L2 regularisation (reg_alpha, reg_lambda) for XGBoost.\n'
     '     Reduce max_depth to 4-6 to constrain model complexity.\n'
     '     Apply early stopping with a validation set during training.'),

    ('Feature Leakage (Real-time)',
     'Replace flow-level rate features with sub-flow windowed aggregates (first 10 packets).\n'
     '     Train a separate early-detection model on forward-packet-only features for\n'
     '     real-time inference; retain the full-flow model for post-hoc forensic analysis.'),

    ('Temporal Drift',
     'Implement Population Stability Index (PSI) monitoring on top-10 SHAP features weekly.\n'
     '     Trigger retraining when PSI > 0.2 on any top-5 feature.\n'
     '     Maintain a rolling 90-day labeled data buffer from production traffic.'),

    ('Adversarial Evasion',
     'Add adversarial training: generate perturbed attack samples by nudging rate features\n'
     '     toward benign ranges, then include them in training with attack labels.\n'
     '     Deploy an OOD (out-of-distribution) detector on SHAP feature space to flag\n'
     '     inputs that fall outside the training manifold.'),

    ('FPR Disparity (Volume Groups)',
     'Apply per-group threshold calibration (as computed above).\n'
     '     Target equalized FNR across volume quartiles given the higher cost of missed\n'
     '     attacks in high-volume segments (DDoS traffic appears there).\n'
     '     Monitor group-specific FPR/FNR in production dashboard monthly.'),

    ('Human-in-the-Loop',
     'Route predictions with P(Attack) in [0.40, 0.70] to analyst review queue\n'
     '     (uncertain zone), rather than auto-blocking. This reduces false positives\n'
     '     on legitimate traffic while preserving recall on high-confidence detections.'),
]

for issue, strategy in mitigations:
    print(f'  [{issue}]')
    print(f'     {strategy}')
    print()

In [ ]:
# --- Save all fairness metrics to CSV ---
df_fair_volume.to_csv('./outputs/step5/fairness_volume.csv', index=False)
df_fair_duration.to_csv('./outputs/step5/fairness_duration.csv', index=False)
df_fair_protocol.to_csv('./outputs/step5/fairness_protocol.csv', index=False)
df_overfit.to_csv('./outputs/step5/overfitting_analysis.csv')

# Save optimal threshold
with open('./models/optimal_threshold.json', 'w') as f:
    json.dump({'binary_threshold': float(opt_threshold)}, f, indent=2)

# Save per-group thresholds
with open('./models/group_thresholds_volume.json', 'w') as f:
    json.dump({k: float(v) for k, v in group_thresholds.items()}, f, indent=2)

print('=== STEP 5 ARTEFACTS SAVED ===')
for root, dirs, files in os.walk('./outputs/step5'):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f'  {fpath:<55} {size_kb:>7.1f} KB')

print()
for fname in ['optimal_threshold.json', 'group_thresholds_volume.json']:
    fpath = f'./models/{fname}'
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'  {fpath:<55} {size_kb:>7.1f} KB')